In [ ]:
pip install rouge-score

In [1]:
import os
os.environ ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
!nvidia-smi

Fri May 24 00:55:24 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.154.05             Driver Version: 535.154.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off | 00000000:01:00.0 Off |                    0 |
| N/A   75C    P0             236W / 250W |  40288MiB / 40960MiB |     98%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

|   1  NVIDIA A100-PCIE-40GB          Off | 00000000:81:00.0 Off |                    0 |
| N/A   72C    P0             252W / 250W |  36051MiB / 40960MiB |    100%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+----------------------+
                                                                                         
+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
|    0   N/A  N/A      2653      G   /usr/libexec/Xorg                            63MiB |
|    0   N

In [ ]:
pip install -U sentence-transformers

In [2]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /home/ramayana/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
from tqdm import tqdm
import csv
import time

In [4]:
from rouge_score import rouge_scorer
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.meteor_score import meteor_score as ms

In [5]:
import pandas as pd
import numpy as np

In [7]:
def calculate_score(text,reference):
    # define the texts to compare
    # tokenize the texts and reference
    text_tokens = word_tokenize(text)
    reference_tokens = word_tokenize(reference)

    # initialize the Rouge scorer
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    # calculate the Rouge 1, Rouge 2, and Rouge L scores
    rouge_scores = scorer.score(text, reference)

    # print the Rouge scores
    rouge_1=rouge_scores['rouge1'].fmeasure
    rouge_2=rouge_scores['rouge2'].fmeasure
    rouge_l=rouge_scores['rougeL'].fmeasure
    
    # calculate the BLEU score
    bleu_score = sentence_bleu([reference_tokens], text_tokens)

    # print the BLEU score
#     print("BLEU:", bleu_score)

    # tokenize the texts for Meteor
    text_tokens = [token.lower() for token in text_tokens]
    reference_tokens = [token.lower() for token in reference_tokens]
    # calculate the Meteor score
    meteor_score = ms([text_tokens], reference_tokens)

    # print the Meteor score
#     print("Meteor:", meteor_score)
    return rouge_1,rouge_2,rouge_l,bleu_score,meteor_score

In [8]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

/home/ramayana/madhan/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
data=pd.read_csv('')

In [10]:
def get_mpnet_embedding(text:str)->list[float]:
    return (model.encode(text))
def get_mpnet_similarity(text1,text2):
    embedding1 = (get_mpnet_embedding(text1))
    embedding2 = (get_mpnet_embedding(text2))
    cosine_sim = np.dot(embedding1, embedding2) / (np.linalg.norm(embedding1) * np.linalg.norm(embedding2))
    return cosine_sim

In [11]:
rouge_1=[]
rouge_2=[]
rouge_l=[]
bleu_score=[]
meteor_score=[]
cos_sim=[]
for i in tqdm(range(len(data))):
    text=data['answer_generated'][i]
    reference=data['answer'][i]
    tupl=calculate_score(text,reference)
    rouge_1.append(tupl[0])
    rouge_2.append(tupl[1])
    rouge_l.append(tupl[2])
    bleu_score.append(tupl[3])
    meteor_score.append(tupl[4])
    cos_sim.append(get_mpnet_similarity(text,reference))
    time.sleep(0.2)

  0%|          | 0/100 [00:00<?, ?it/s]/home/ramayana/madhan/miniconda3/lib/python3.10/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/ramayana/madhan/miniconda3/lib/python3.10/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/ramayana/madhan/miniconda3/lib/python3.10/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps o

In [12]:
data['rouge_1']=rouge_1
data['rouge_2']=rouge_2
data['rouge_l']=rouge_l
data['meteor_score']=meteor_score
data['similarity']=cos_sim

In [13]:
data.head(2)

,question,answer,answer_generated,context,type,score,Unnamed: 6,rouge_1,rouge_2,rouge_l,meteor_score,similarity
0,Dear Sir/mam Please Assist Me How And Where Sh...,"Dear Client, Anticipatory By Its Very Nature I...",If you wish to apply for the cancellation of a...,Summary: The Supreme Court of India has dismis...,Anticipatory_bail,3,NaN,0.288660,0.062176,0.170103,0.242336,0.785415
1,Can A Person Get Bail After Being Sentenced By...,Dear Client If The Convicted Person Files An A...,"Yes, a person can still seek bail after being ...",Summary Article: The case of Gudikanti Narasim...,Anticipatory_bail,2,NaN,0.421053,0.177515,0.304094,0.229209,0.800049


In [60]:
# new_cols=['title','question','ground_truth','answer_generated','rouge_1','rouge_2','rouge_l','meteor_score','similarity','score']

In [61]:
# data=data[new_cols]

In [14]:
data.to_csv('')